# Actividad 02 — Fusión de las 5 fuentes en el Dataset Maestro v2

**Fase:** 2 — Ingeniería de Características Multimodal  
**Dominio:** Predicción de producción de limón (Sutil y Dulce), 2016-2025

---

## Objetivo

Integrar las **5 fuentes ya limpias y validadas** en un dataset maestro único
a nivel **nacional-mensual** (120 filas, 2016-01 a 2025-12), en **dos versiones
separadas** (Sutil y Dulce) que comparten las variables de clima, INDECI y NLP.

## Fuentes

| Fuente | Archivo | Nivel | Columnas |
|---|---|---|---|
| Producción Sutil | `v2_reentrenamiento/data/interim/limon_sutil_nacional_mensual.csv` | nacional | produccion, precio, n_provincias |
| Producción Dulce | `v2_reentrenamiento/data/interim/limon_dulce_nacional_mensual.csv` | nacional | produccion, precio, n_provincias |
| Clima NASA POWER | `v2_reentrenamiento/data/raw/nasa_power/por_provincia/clima_nasa_power_2016_2025.csv` | **provincia** (109) | T2M, T2M_MAX, WS2M, PRECTOTCORR, RH2M |
| Emergencias INDECI | `v2_reentrenamiento/data/interim/indeci_limpio_2016_2025.csv` | **provincia** (107) | emergencias, afectados, hectáreas |
| Sentimiento NLP | `v2_reentrenamiento/data/interim/noticias/sentimiento_mensual_2016_2025.csv` | nacional | avg_sentiment, n_noticias |

## Criterio de agregación provincial → nacional

**Producción ponderada por provincia** (mismo criterio aprobado para
`PRECIO_CHACRA`): el clima y las emergencias de las provincias que más producen
limón deben pesar más en el agregado nacional. Para cada mes:

$$\bar{x}_{nacional,m} = \frac{\sum_{p} x_{p,m} \cdot produccion_{p,m}}{\sum_{p} produccion_{p,m}}$$

- **Pesos:** producción total de limón (Sutil + Dulce) por provincia y mes
  (única agregación nacional compartida por ambas versiones del maestro).
- **NASA:** 5 variables climáticas, presentes en las 109 provincias.
- **INDECI:** 6 variables de emergencia; las 2 provincias Dulce sin registro
  (CARHUAZ, YUNGAY) no contribuyen al numerador ni denominador.
- **NLP:** ya es nacional mensual; se integra directo.

> Se usan pesos **por mes** (varían mes a mes con la producción provincial).

## Salidas

- `v2_reentrenamiento/data/processed/master_dataset_sutil_v2.csv` (18 columnas, 120 filas)
- `v2_reentrenamiento/data/processed/master_dataset_dulce_v2.csv` (18 columnas, 120 filas)

---


## 1. Configuración inicial


In [ ]:
import os, warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', None)

while not os.path.exists('v2_reentrenamiento/data/interim'):
    os.chdir('..')
print('Raiz del proyecto:', os.getcwd())

V2 = 'v2_reentrenamiento/data'
PATH_SUTIL_NAC  = f'{V2}/interim/limon_sutil_nacional_mensual.csv'
PATH_DULCE_NAC  = f'{V2}/interim/limon_dulce_nacional_mensual.csv'
PATH_SUTIL_PROV = f'{V2}/interim/limon_sutil_provincia.csv'
PATH_DULCE_PROV = f'{V2}/interim/limon_dulce_provincia.csv'
PATH_NASA       = f'{V2}/raw/nasa_power/por_provincia/clima_nasa_power_2016_2025.csv'
PATH_INDECI     = f'{V2}/interim/indeci_limpio_2016_2025.csv'
PATH_SENT       = f'{V2}/interim/noticias/sentimiento_mensual_2016_2025.csv'

OUT_SUTIL = 'v2_reentrenamiento/data/processed/master_dataset_sutil_v2.csv'
OUT_DULCE = 'v2_reentrenamiento/data/processed/master_dataset_dulce_v2.csv'

VARS_CLIMA  = ['T2M', 'T2M_MAX', 'WS2M', 'PRECTOTCORR', 'RH2M']
VARS_INDECI = ['num_emergencias', 'personas_afectadas', 'personas_damnificadas',
               'total_afectados', 'hectareas_cultivo_perdidas', 'hectareas_cultivo_afectadas']
VARS_NLP    = ['avg_sentiment', 'n_noticias']
VARS_SHARED = VARS_CLIMA + VARS_INDECI + VARS_NLP


## 2. Carga de las 5 fuentes


In [ ]:
sutil_nac = pd.read_csv(PATH_SUTIL_NAC, encoding='utf-8-sig')
dulce_nac = pd.read_csv(PATH_DULCE_NAC, encoding='utf-8-sig')
sutil_prov = pd.read_csv(PATH_SUTIL_PROV, encoding='utf-8-sig')
dulce_prov = pd.read_csv(PATH_DULCE_PROV, encoding='utf-8-sig')
nasa = pd.read_csv(PATH_NASA, encoding='utf-8-sig')
indeci = pd.read_csv(PATH_INDECI, encoding='utf-8-sig')
indeci = indeci.rename(columns={'anio': 'año'})   # normalizar clave temporal
sent = pd.read_csv(PATH_SENT, encoding='utf-8-sig')

for n, d in [('sutil_nac', sutil_nac), ('dulce_nac', dulce_nac), ('sutil_prov', sutil_prov),
             ('dulce_prov', dulce_prov), ('nasa', nasa), ('indeci', indeci), ('sent', sent)]:
    print(f'{n:10s}: {d.shape}')

nasa['DATE'] = pd.to_datetime(nasa['DATE'])
nasa['año'] = nasa['DATE'].dt.year
nasa['mes'] = nasa['DATE'].dt.month

meses = {'sutil_nac': sutil_nac, 'dulce_nac': dulce_nac, 'sent': sent,
         'nasa': nasa, 'indeci': indeci}
for n, d in meses.items():
    print(f'{n:10s}: {d.groupby(["año","mes"]).ngroups} meses únicos, '
          f'años {d["año"].min()}-{d["año"].max()}, mes {d["mes"].min()}-{d["mes"].max()}')


## 3. Pesos de producción provincial por mes


In [ ]:
# Peso = producción total de limón (Sutil + Dulce) por provincia y mes
pesos = pd.concat([
    sutil_prov[['año', 'mes', 'departamento', 'provincia', 'produccion_t']],
    dulce_prov[['año', 'mes', 'departamento', 'provincia', 'produccion_t']],
], ignore_index=True)
pesos = pesos.groupby(['año', 'mes', 'departamento', 'provincia'],
                      as_index=False)['produccion_t'].sum()

print('Pesos (provincia-mes):', pesos.shape)
print('Provincias con peso:', pesos['provincia'].nunique())
tot_mes = pesos.groupby(['año', 'mes'])['produccion_t'].sum()
print('Meses con produccion total <= 0:', int((tot_mes <= 0).sum()))

pesos_key = set(map(tuple, pesos[['departamento', 'provincia']].drop_duplicates().values))
nasa_key  = set(map(tuple, nasa[['departamento', 'provincia']].drop_duplicates().values))
indeci_key = set(map(tuple, indeci[['departamento', 'provincia']].drop_duplicates().values))
print()
print('Productoras SIN NASA  :', sorted(pesos_key - nasa_key))
print('Productoras SIN INDECI:', sorted(pesos_key - indeci_key))


## 4. Agregación nacional ponderada — NASA POWER


In [ ]:
def agregar_nacional_ponderado(fuente, pesos, var_cols):
    '''Media produccion-ponderada por mes: sum(x*prod) / sum(prod).'''
    m = fuente.merge(pesos, on=['año', 'mes', 'departamento', 'provincia'], how='left')
    prod = m['produccion_t'].fillna(0.0)
    numerador = pd.DataFrame({v: m[v] * prod for v in var_cols})
    numerador['produccion_t'] = prod
    numerador['año'] = m['año']
    numerador['mes'] = m['mes']
    gnum = numerador.groupby(['año', 'mes']).sum()
    gden = gnum['produccion_t'].replace(0, np.nan)
    out = gnum[var_cols].divide(gden, axis=0)
    out = out.reset_index().sort_values(['año', 'mes']).reset_index(drop=True)
    n_prov = fuente[['departamento', 'provincia']].drop_duplicates().shape[0]
    return out, n_prov

clima_nac, n_prov_clima = agregar_nacional_ponderado(nasa, pesos, VARS_CLIMA)
clima_nac[VARS_CLIMA] = clima_nac[VARS_CLIMA].round(4)

print(f'Aggregación NASA: {clima_nac.shape} | provincias con registro: {n_prov_clima}')
print()
print('Primeros 5 meses:')
print(clima_nac.head(5).round(4).to_string(index=False))
print()
print('Rangos de validación (físicos):')
print('  T2M         ∈ [{:.1f}, {:.1f}] °C'.format(clima_nac['T2M'].min(), clima_nac['T2M'].max()))
print('  T2M_MAX     ∈ [{:.1f}, {:.1f}] °C'.format(clima_nac['T2M_MAX'].min(), clima_nac['T2M_MAX'].max()))
print('  WS2M        ∈ [{:.2f}, {:.2f}] m/s'.format(clima_nac['WS2M'].min(), clima_nac['WS2M'].max()))
print('  PRECTOTCORR ∈ [{:.2f}, {:.2f}] mm/dia'.format(clima_nac['PRECTOTCORR'].min(), clima_nac['PRECTOTCORR'].max()))
print('  RH2M        ∈ [{:.1f}, {:.1f}] %'.format(clima_nac['RH2M'].min(), clima_nac['RH2M'].max()))


## 5. Agregación nacional ponderada — INDECI


In [ ]:
indeci_nac, n_prov_ind = agregar_nacional_ponderado(indeci, pesos, VARS_INDECI)
indeci_nac[VARS_INDECI] = indeci_nac[VARS_INDECI].round(4)

print(f'Aggregación INDECI: {indeci_nac.shape} | provincias con registro: {n_prov_ind}')
print()
print('Primeros 5 meses:')
print(indeci_nac.head(5).round(4).to_string(index=False))
print()
print('Meses sin emergencias (suma 0):', int((indeci_nac[VARS_INDECI].sum(axis=1) == 0).sum()))
print('Meses cubiertos: 120' if len(indeci_nac) == 120 else f'ERROR: solo {len(indeci_nac)} meses')


## 6. Fusión final — Dataset Maestro v2 (Sutil y Dulce)


In [ ]:
def construir_maestro(nac, sufijo_cultivo):
    df = nac.rename(columns={
        'produccion_t_nacional': f'produccion_t_{sufijo_cultivo}',
        'precio_chacra_kg_nacional': f'precio_chacra_kg_{sufijo_cultivo}',
        'n_provincias_reportando': f'n_provincias_{sufijo_cultivo}',
    })
    df = df.merge(clima_nac, on=['año', 'mes'], how='left')
    df = df.merge(indeci_nac, on=['año', 'mes'], how='left')
    df = df.merge(sent, on=['año', 'mes'], how='left')
    df = df.sort_values(['año', 'mes']).reset_index(drop=True)
    return df

maestro_sutil = construir_maestro(sutil_nac, 'sutil')
maestro_dulce = construir_maestro(dulce_nac, 'dulce')

COLUMNAS = {
    'sutil': ['año', 'mes', 'produccion_t_sutil', 'precio_chacra_kg_sutil',
              'n_provincias_sutil'] + VARS_SHARED,
    'dulce': ['año', 'mes', 'produccion_t_dulce', 'precio_chacra_kg_dulce',
              'n_provincias_dulce'] + VARS_SHARED,
}
maestro_sutil = maestro_sutil[COLUMNAS['sutil']]
maestro_dulce = maestro_dulce[COLUMNAS['dulce']]

print('MAESTRO SUTIL:', maestro_sutil.shape)
print('MAESTRO DULCE:', maestro_dulce.shape)
print()
print('Columnas SUTIL:', list(maestro_sutil.columns))
print('Columnas DULCE:', list(maestro_dulce.columns))


### Guardar


In [ ]:
os.makedirs(os.path.dirname(OUT_SUTIL), exist_ok=True)
maestro_sutil.to_csv(OUT_SUTIL, index=False, encoding='utf-8-sig')
maestro_dulce.to_csv(OUT_DULCE, index=False, encoding='utf-8-sig')
print('Guardado:', OUT_SUTIL, f'({maestro_sutil.shape[0]} filas)')
print('Guardado:', OUT_DULCE, f'({maestro_dulce.shape[0]} filas)')


## 7. Verificación — 120 filas, 0 nulos, columnas esperadas


In [ ]:
def verificar_maestro(df, nombre, esperadas):
    print('=' * 70)
    print(nombre, '|', df.shape)
    print('-' * 70)
    ok_filas = len(df) == 120
    print('1) Exactamente 120 filas:', ok_filas)
    print('   Rango:', f'{df["año"].min()}-{df["mes"].min()} -> '
                      f'{df["año"].max()}-{df["mes"].max()}',
          '| meses únicos:', df.groupby(['año', 'mes']).ngroups)
    nulos = int(df.isna().sum().sum())
    print('2) Nulos totales:', nulos)
    nu = df.isna().sum()
    if (nu > 0).any():
        print('   Nulos por columna:'); print(nu[nu > 0].to_string())
    faltantes = [c for c in esperadas if c not in df.columns]
    print('3) Columnas esperadas presentes:', not faltantes)
    if faltantes: print('   FALTANTES:', faltantes)
    sobrantes = [c for c in df.columns if c not in esperadas]
    if sobrantes: print('   Columnas extra (revisar):', sobrantes)
    return ok_filas and nulos == 0 and not faltantes

ok_s = verificar_maestro(maestro_sutil, 'MAESTRO SUTIL v2', COLUMNAS['sutil'])
ok_d = verificar_maestro(maestro_dulce, 'MAESTRO DULCE v2', COLUMNAS['dulce'])
print()
print('RESULTADO GENERAL:', 'PASA ✓' if (ok_s and ok_d) else 'FALLA ✗')


## 8. Primeras y últimas filas de cada maestro


In [ ]:
print('=== MAESTRO SUTIL v2 — primeras 3 filas ===')
print(maestro_sutil.head(3).round(4).to_string(index=False))
print()
print('=== MAESTRO SUTIL v2 — últimas 3 filas ===')
print(maestro_sutil.tail(3).round(4).to_string(index=False))


In [ ]:
print('=== MAESTRO DULCE v2 — primeras 3 filas ===')
print(maestro_dulce.head(3).round(4).to_string(index=False))
print()
print('=== MAESTRO DULCE v2 — últimas 3 filas ===')
print(maestro_dulce.tail(3).round(4).to_string(index=False))


## 9. Sanidad cruzada rápida


In [ ]:
compartidas_s = maestro_sutil[VARS_SHARED]
compartidas_d = maestro_dulce[VARS_SHARED]
print('clima+INDECI+NLP idéntico entre Sutil y Dulce (compartido):',
      compartidas_s.equals(compartidas_d))
print()
print('T2M > 0 en 120/120 meses:', bool((maestro_sutil['T2M'] > 0).all()))
print('RH2M ∈ [0,100]:', bool(maestro_sutil['RH2M'].between(0, 100).all()))
print('n_provincias > 0 en 120/120:', bool((maestro_sutil['n_provincias_sutil'] > 0).all()))
print('Corr(T2M, prod_sutil): {:.3f}'.format(
    maestro_sutil['T2M'].corr(maestro_sutil['produccion_t_sutil'])))
print('Corr(avg_sentiment, prod_sutil): {:.3f}'.format(
    maestro_sutil['avg_sentiment'].corr(maestro_sutil['produccion_t_sutil'])))
print()
print('=== FIN ACTIVIDAD 02 (v2) ===')
